# RQ2 on the Verify-and-Ask notes (201 simulated cases)

**What this notebook does.** It takes the notes produced by verify-and-ask (RQ1): verified S/O, plus an A/P that RQ1 left untouched. It then:
1. **trains the verifier**, a small model (Qwen2.5-3B) taught by example to answer two questions: *"is this statement supported by this text?"* and *"do these patient facts justify this conclusion?"*;
2. **checks every A/P statement through its evidence chain** (blocks 4–11 of the block diagram) and compares this with the old "was it said?" checkers;
3. **signs off** each note (block 12, simulated doctor: flagged lines held back) and **writes a checked patient summary** (blocks 13–14);
4. saves **a few CSV/JSON files** with the results of every block.

**How to use it.** `Runtime → Change runtime type → A100 GPU` (an L4 also works, just slower), then `Runtime → Run all`.
Everything is saved on Google Drive, in `MyDrive/tokenizers/rq2_evidence_chain/`, as it goes. **After a disconnect, simply run all cells again**: finished work is skipped and training continues from its last checkpoint. The last cell releases the GPU.

| Part | What runs | Diagram blocks | A100 (estimate) | L4 (estimate) |
|---|---|---|---|---|
| Setup | packages, Qwen3-14B generator | — | 10–15 min / session | 15–20 min / session |
| A | verifier training data + training | verifier | 3–4.5 h | 6–8 h |
| B | data, planted errors, dev + test runs, metrics | 0, 4–11 | 5.5–6.5 h | 10–13 h |
| C | sign-off + patient summaries | 12–14 | ~1 h | ~2 h |
| D | export results, release GPU | — | 1 min | 1 min |

Estimates only: the code sends one request at a time, so real times can be 50% higher or lower.

**Which cases.** The 201 cases are split as in RQ1, and the 65 cases built from ACI-Bench *train* encounters are left out, because the verifier learns from those patients. That leaves **136 scored cases**: `va_dev` (26, only used to choose the flag threshold) and `va_test` (110, the reported results).

**Conditions compared** (the same A/P statements, different checkers):
**U1** = uMedSum's DeBERTa, "was it said?" (baseline) · **U2** = our verifier, same "was it said?" question · **U3** = evidence chain without MedlinePlus · **U4** = evidence chain with MedlinePlus (**our method, the diagram**).

## 0 · Settings
Change these only if your folders differ. `EXISTING_ADAPTER` lets you reuse an already-trained verifier and skip most of Part A. Leave it empty to train from scratch.

In [1]:
# ---- where things live (Google Drive) --------------------------------------------------------------
DRIVE    = "/content/drive/MyDrive"
REPO_URL = "https://github.com/DoshikRahul/tokenizers.git"
REPO     = "/content/tokenizers"                 # fresh copy of the code from GitHub on the Colab machine (step 2);
                                                 # the (possibly outdated) copy on Drive is never used
RQ2      = f"{REPO}/rq2_section_aware"           # RQ2 code
BASE     = f"{DRIVE}/tokenizers/rq2_evidence_chain"   # EVERYTHING this notebook saves goes in here
WORK     = f"{BASE}/work"                        # working files and caches (needed for resuming)
RESULTS  = f"{BASE}/results"                     # the final CSV / JSON files
LOGS     = f"{BASE}/logs"                        # generator and smoke-test logs
SCRIPTS  = f"{BASE}/scripts"                     # the five helper scripts written in step 3

# ---- choices -----------------------------------------------------------------------------------------
K_REF            = 3      # MedlinePlus passages kept per piece (block 9); chosen on ACI-Bench dev (report §6.3)
CONDITIONS       = ["U4", "U3", "U2", "U1"]      # U4 first: it extracts the facts and links that U3 reuses
EXISTING_ADAPTER = ""     # e.g. f"{DRIVE}/tokenizers/rq2_section_aware/models/verifier_qlora" ("" = train here)
RUN_SMOKE_TEST   = True   # 1-minute offline test of the code, first session only
DISCONNECT_AT_END = True  # release the GPU when everything is finished

## 1 · Check the GPU
Picks the settings for the GPU Colab gave you. Qwen3-14B (the generator) and the small verifier share one GPU, so the generator gets only part of the memory.

In [2]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "No GPU. Runtime -> Change runtime type -> A100 GPU (or L4)."
gpu_name, gpu_mem = gpu.split(",")[0].strip(), int(gpu.split(",")[1]) / 1024
print(f"GPU: {gpu_name}, {gpu_mem:.0f} GB")

if "A100" in gpu_name and gpu_mem > 60:      # A100 80 GB: full-precision generator
    PROFILE = dict(model="Qwen/Qwen3-14B",     util=0.50, max_len=16384)
elif "A100" in gpu_name or "H100" in gpu_name:  # A100 40 GB: 4-bit generator (official AWQ build)
    PROFILE = dict(model="Qwen/Qwen3-14B-AWQ", util=0.55, max_len=16384)
elif "L4" in gpu_name:                         # L4 24 GB: 4-bit generator, shorter context
    PROFILE = dict(model="Qwen/Qwen3-14B-AWQ", util=0.55, max_len=12288)
else:
    raise SystemExit(f"{gpu_name} is too small for this notebook: use an A100 or L4 runtime.")
print("profile:", PROFILE)

GPU: NVIDIA A100-SXM4-40GB, 40 GB
profile: {'model': 'Qwen/Qwen3-14B-AWQ', 'util': 0.55, 'max_len': 16384}


## 2 · Google Drive and the code
Mounts Drive (approve the pop-up). Then takes a **fresh copy of the code from GitHub** onto the Colab machine. That takes about a minute each session, and the copy of the repo on your Drive is never used or changed.
**Resume safety:** the first session saves which GitHub commit it used (`rq2_evidence_chain/repo_commit.txt`), and every later session uses that same commit, even if someone pushes to the repo in between.
All outputs go to `MyDrive/tokenizers/rq2_evidence_chain/`.

In [3]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(BASE, exist_ok=True)
if not os.path.exists(f"{REPO}/.git"):
    !git clone -q {REPO_URL} {REPO}
pin = f"{BASE}/repo_commit.txt"
if os.path.exists(pin):                       # later sessions: exactly the code the first session used
    commit = open(pin).read().strip()
    !cd {REPO} && git checkout -q {commit}
else:                                         # first session: remember the commit
    commit = !cd {REPO} && git rev-parse HEAD
    open(pin, "w").write(commit[0])
!cd {REPO} && git log --oneline -1
for d in ("aci_train/data", "aci_train/runs", "aci_train/reports", "va/data", "va/runs", "va/reports",
          "reference_corpus", "models", "final"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
for d in (RESULTS, LOGS, SCRIPTS):
    os.makedirs(d, exist_ok=True)
%cd {RQ2}

Mounted at /content/drive
04194dc (HEAD -> main, origin/main, origin/HEAD) Baseline: remove old 110-case leftovers and cache files; fix .gitignore encoding
/content/tokenizers/rq2_section_aware


## 3 · Helper scripts
Five small scripts this run needs on top of the RQ2 code in the repository. Each cell writes one script into `rq2_evidence_chain/scripts/` (the repo's own code is not touched), so the notebook doesn't depend on them being pushed to GitHub. You don't need to read them.

| Script | Block(s) | Does |
|---|---|---|
| `build_dataset_va.py` | 0, 4 | turns the verify-and-ask notes into RQ2's input files and makes the splits |
| `train_verifier_r.py` | verifier | the training script with checkpoints every 25 steps and automatic resume |
| `signoff.py` | 12 | doctor-review note + signed note per case |
| `patient_summary.py` | 13, 14 | plain-language summary from the signed note, each sentence checked; H3 comparison |
| `export_results.py` | all | the final CSV / JSON files |

In [4]:
%%writefile {SCRIPTS}/build_dataset_va.py
# blocks 0 + 4: verify-and-ask notes -> RQ2 input files and splits
"""Build the RQ2 statement sets from the Verify-and-Ask (RQ1) outputs instead of ACI-Bench reference notes.

Input  : verify_and_ask_pipeline/Stage23_pipeline/output/stage23_output.json  (201 cases)
         - conversation : the simulated intake conversation (block 0 in the diagram)
         - stage1_ap    : the Assessment / Plan written by the RQ1 note writer, untouched by RQ1 (block 4 input)
Output : the same file types build_dataset.py writes, so every other RQ2 script works unchanged:
         encounters_<split>.jsonl, statements_<split>.jsonl, audit_valid_<split>.csv, va_manifest.json

Splits (use a separate RQ2_DATA_DIR / RQ2_RUNS_DIR / RQ2_REPORT_DIR, see the bottom of this docstring):
  va_dev        RQ1 dev cases whose ACI-Bench source is NOT train      -> tuning (thresholds)
  va_test       RQ1 test cases whose ACI-Bench source is NOT train     -> reported results
  va_trainorig  cases built from ACI-Bench TRAIN encounters (both RQ1 splits). The verifier was trained on
                these patients, so they are run only to produce final notes, never scored.

Differences from build_dataset.py (ACI-Bench):
  * evidence = the simulated conversation. It contains no spoken diagnosis or plan (the intake only takes the
    history), so it is already "pre-A/P": turns_preap = turns and mask_ap.py is not needed.
  * statements = one per sentence of each A/P bullet; a short field label such as
    "Clinical problems or diagnoses:" is removed; placeholder lines such as
    "No explicit management or follow-up plan was documented" are excluded.
  * gold_label = "valid" for every statement, as for ACI reference statements; audit_valid_<split>.csv lets the
    team mark wrong ones (valid_ok = 0), exactly as before.

Usage (from rq2_section_aware/):
  export RQ2_DATA_DIR=data_out_va RQ2_RUNS_DIR=runs_va RQ2_REPORT_DIR=reports_va
  python build_dataset_va.py
  # defaults: --va-json ../verify_and_ask_pipeline/Stage23_pipeline/output/stage23_output.json
  #           --prepared ../verify_and_ask_pipeline/stage1_pipeline/stage1_output/prepared_dataset.json
Keep the three directories separate from the ACI-Bench run: feature caches are stored per encounter id, and
D2Nxxx would otherwise reuse features extracted from the ORIGINAL dialogue.
"""
import argparse
import json
import re
from collections import Counter
from pathlib import Path

import config as C
from build_dataset import classify_free_sentence
from common import parse_aci_dialogue, read_csv, split_sentences, strip_bullet, write_csv, write_jsonl

PLACEHOLDER = re.compile(
    r"^\s*no (explicit |specific |clear )?(management |follow[- ]?up |treatment |diagnostic )*(or follow[- ]?up )?"
    r"(plan|plans|treatment|diagnos\w*|medications?)\b.*\b(was|were|is|are|has been|have been)\b.*"
    r"\b(documented|mentioned|discussed|established|provided|stated)\b"
    r"|\bnot (explicitly )?(documented|mentioned|discussed|established) in the conversation\b"
    r"|^\s*(none|n/?a|not applicable|not documented|unknown)\.?\s*$",
    re.IGNORECASE)
FIELD_LABEL = re.compile(r"^([A-Za-z][A-Za-z /&(),-]{2,80}):\s+(.+)$")


def ap_statements(case_id, ap):
    rows = []
    for section, key in (("Assessment", "assessment"), ("Plan", "plan")):
        kind = "A" if section == "Assessment" else "P"
        for raw in (ap.get(key) or "").splitlines():
            line = strip_bullet(raw.strip())
            if not line:
                continue
            m = FIELD_LABEL.match(line)
            if m and len(m.group(1).split()) <= 10:   # "Clinical problems or diagnoses: ..." -> keep the content
                line = m.group(2).strip()
            for sent in split_sentences(line):
                if len(sent.split()) < 2:
                    continue
                placeholder = bool(PLACEHOLDER.search(sent))
                if placeholder:
                    stype = "placeholder"
                else:
                    stype, _ = classify_free_sentence(sent, kind)
                rows.append(dict(text=sent, section=section, stmt_type=stype, exclude=placeholder))
    out = []
    for i, r in enumerate(rows, 1):
        out.append(dict(statement_id=f"{case_id}_V{i:02d}", encounter_id=case_id, claim=r["text"], text=r["text"],
                        stmt_type=r["stmt_type"], section=r["section"], problem="", source_header="va_stage1_ap",
                        origin="va_generated", gold_label="valid", error_type="", exclude=r["exclude"]))
    return out


def aci_origin():
    """encounter_id -> ACI-Bench split file it comes from (train / valid / test1 ...)."""
    origin = {}
    for name, fname in C.ACI_SPLITS.items():
        for r in read_csv(C.ACI_DIR / fname):
            origin[r["encounter_id"]] = name
    return origin


def reference_text(ref):
    return "\n\n".join(f"{k.upper()}\n{(ref.get(k) or '').strip()}" for k in ("subjective", "objective",
                                                                              "assessment", "plan"))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--va-json", default=str(C.REPO / "verify_and_ask_pipeline" / "Stage23_pipeline" / "output"
                                             / "stage23_output.json"))
    ap.add_argument("--prepared", default=str(C.REPO / "verify_and_ask_pipeline" / "stage1_pipeline"
                                              / "stage1_output" / "prepared_dataset.json"))
    a = ap.parse_args()

    if C.DATA_DIR.resolve() == (C.ROOT / "data_out").resolve():
        raise SystemExit("Set RQ2_DATA_DIR (and RQ2_RUNS_DIR, RQ2_REPORT_DIR) to separate folders first, e.g.\n"
                         "  export RQ2_DATA_DIR=data_out_va RQ2_RUNS_DIR=runs_va RQ2_REPORT_DIR=reports_va")
    records = [r for r in json.loads(Path(a.va_json).read_text(encoding="utf-8"))["records"]
               if r.get("status") == "success"]
    refs = {r["case_id"]: r["reference_soap"]
            for r in json.loads(Path(a.prepared).read_text(encoding="utf-8"))["records"]}
    origin = aci_origin()

    buckets = {"va_dev": ([], []), "va_test": ([], []), "va_trainorig": ([], [])}
    manifest = []
    for r in records:
        cid = r["case_id"]
        src = origin.get(cid, "unknown")
        split = "va_trainorig" if src == "train" else ("va_dev" if r["split"] == "dev" else "va_test")
        turns = parse_aci_dialogue(r["conversation"])
        if not turns:
            raise SystemExit(f"{cid}: conversation has no 'Doctor:' / 'Patient:' turns")
        enc = dict(encounter_id=cid, dataset="simulated_intake", split=split, rq1_split=r["split"],
                   aci_source_split=src, in_verifier_training=src == "train",
                   turns=turns, turns_preap=[dict(t) for t in turns],
                   mask_log="not masked: the simulated intake conversation contains no spoken diagnosis or plan",
                   reference_note=reference_text(refs.get(cid, {})))
        stmts = ap_statements(cid, r["stage1_ap"])
        buckets[split][0].append(enc)
        buckets[split][1].extend(stmts)
        kept = [s for s in stmts if not s["exclude"]]
        manifest.append(dict(case_id=cid, rq2_split=split, rq1_split=r["split"], aci_source_split=src,
                             n_turns=len(turns), n_statements=len(kept),
                             n_assessment=sum(s["section"] == "Assessment" for s in kept),
                             n_plan=sum(s["section"] == "Plan" for s in kept),
                             n_placeholders=len(stmts) - len(kept)))

    C.DATA_DIR.mkdir(parents=True, exist_ok=True)
    for split, (encs, stmts) in buckets.items():
        write_jsonl(C.DATA_DIR / f"encounters_{split}.jsonl", encs)
        write_jsonl(C.DATA_DIR / f"statements_{split}.jsonl", stmts)
        write_csv(C.DATA_DIR / f"audit_valid_{split}.csv",
                  [dict(s, valid_ok="", section_ok="", annotator="", notes="") for s in stmts if not s["exclude"]],
                  ["statement_id", "encounter_id", "section", "stmt_type", "problem", "claim",
                   "valid_ok", "section_ok", "annotator", "notes"])
        kept = [s for s in stmts if not s["exclude"]]
        print(f"[{split}] encounters={len(encs)} statements={len(kept)} "
              f"(placeholders excluded: {len(stmts) - len(kept)})")
        print("   section:", dict(Counter(s["section"] for s in kept)))
        no_plan = [e["encounter_id"] for e in encs
                   if not any(s["encounter_id"] == e["encounter_id"] and s["section"] == "Plan" for s in kept)]
        print(f"   encounters with no Plan statement: {len(no_plan)}")
    (C.DATA_DIR / "va_manifest.json").write_text(json.dumps(manifest, indent=1), encoding="utf-8")
    print(f"wrote {C.DATA_DIR}/ (encounters, statements, audit sheets, va_manifest.json)")


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/tokenizers/rq2_evidence_chain/scripts/build_dataset_va.py


In [5]:
%%writefile {SCRIPTS}/train_verifier_r.py
# verifier training with checkpoints and automatic resume
"""Resumable copy of train_verifier.py (used by the RQ2 notebook).

Changes: saves a checkpoint every --save-steps steps and resumes from the newest checkpoint in --out after a
disconnect; skips training if a finished adapter is already there; writes the held-out accuracy to
<out>/heldout_accuracy.json. Everything else is identical to train_verifier.py.

QLoRA fine-tuning of the verifier (Qwen2.5-3B-Instruct, LoRA rank 16) on {"messages", "target"} JSONL.

Inputs use the SAME prompt builders as inference (prompts.py), so label probabilities at test time
are meaningful. Loss is computed on the label tokens (+ end-of-turn) only.

  python train_verifier.py --data data_out/chain_train.jsonl --out models/verifier_qlora
  python train_verifier.py --data data_out/chain_train.jsonl data_out/so_train.jsonl --out models/verifier_qlora
  # 1080 Ti without 4-bit support in bitsandbytes: add --no-4bit (plain LoRA on fp16 weights, ~7 GB)

S/O pairs (if you build them) must look like: {"messages": prompts.entail_messages(evidence, claim), "target": "supported"}.
Encounters are split 90/10 into train / held-out (by encounter_id) and held-out label accuracy is printed.
"""
import argparse
import random

import config as C
from common import read_jsonl


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", nargs="+", required=True)
    ap.add_argument("--out", default=str(C.ROOT / "models" / "verifier_qlora"))
    ap.add_argument("--base", default=C.VERIFIER_BASE_MODEL)
    ap.add_argument("--epochs", type=float, default=2)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--rank", type=int, default=16)
    ap.add_argument("--max-len", type=int, default=3072)
    ap.add_argument("--grad-accum", type=int, default=16)
    ap.add_argument("--no-4bit", action="store_true")
    ap.add_argument("--save-steps", type=int, default=25)
    a = ap.parse_args()

    import glob
    import json
    import os
    done_flag = os.path.join(a.out, "adapter_model.safetensors")
    if os.path.exists(done_flag) and os.path.exists(os.path.join(a.out, "heldout_accuracy.json")):
        print("adapter already trained ->", a.out)
        return
    import torch
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer,
                              TrainingArguments)

    rows = [r for p in a.data for r in read_jsonl(p)]
    encs = sorted({r.get("encounter_id", str(i)) for i, r in enumerate(rows)})
    random.Random(C.SEED).shuffle(encs)
    held = set(encs[: max(1, len(encs) // 10)])
    train = [r for i, r in enumerate(rows) if r.get("encounter_id", str(i)) not in held]
    dev = [r for i, r in enumerate(rows) if r.get("encounter_id", str(i)) in held]
    print(f"train {len(train)}  held-out {len(dev)}  labels {sorted({r['target'] for r in rows})}")

    tok = AutoTokenizer.from_pretrained(a.base)
    tok.pad_token = tok.pad_token or tok.eos_token
    end = tok.eos_token if "<|im_end|>" not in tok.get_vocab() else "<|im_end|>"

    def encode(r):
        prompt = tok.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=True)
        p = tok(prompt, add_special_tokens=False)["input_ids"]
        t = tok(r["target"] + end, add_special_tokens=False)["input_ids"]
        if len(p) + len(t) > a.max_len:  # same truncation rule as verifier.HFVerifier
            keep = a.max_len - len(t)
            p = p[:64] + p[-(keep - 64):]
        return {"input_ids": p + t, "labels": [-100] * len(p) + t}

    train_ds = [encode(r) for r in train]

    def collate(batch):
        n = max(len(b["input_ids"]) for b in batch)
        ids = torch.full((len(batch), n), tok.pad_token_id)
        lab = torch.full((len(batch), n), -100)
        att = torch.zeros((len(batch), n), dtype=torch.long)
        for i, b in enumerate(batch):
            L = len(b["input_ids"])
            ids[i, :L], lab[i, :L], att[i, :L] = torch.tensor(b["input_ids"]), torch.tensor(b["labels"]), 1
        return {"input_ids": ids, "labels": lab, "attention_mask": att}

    kw = {"device_map": "auto"}
    if a.no_4bit:
        kw["torch_dtype"] = torch.float16
    else:
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=torch.float16,
                                                       bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(a.base, **kw)
    if not a.no_4bit:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    else:
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(r=a.rank, lora_alpha=2 * a.rank, lora_dropout=0.05, task_type="CAUSAL_LM",
                                             target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                                             "gate_proj", "up_proj", "down_proj"]))
    model.print_trainable_parameters()
    import inspect
    # transformers 5.x removed warmup_ratio; there a float < 1 in warmup_steps means the same ratio
    warm = ({"warmup_ratio": 0.03} if "warmup_ratio" in inspect.signature(TrainingArguments.__init__).parameters
            else {"warmup_steps": 0.03})
    args = TrainingArguments(output_dir=a.out, **warm, per_device_train_batch_size=1, gradient_accumulation_steps=a.grad_accum,
                             num_train_epochs=a.epochs, learning_rate=a.lr, lr_scheduler_type="cosine",
                             logging_steps=10, save_strategy="steps", save_steps=a.save_steps, save_total_limit=2,
                             fp16=True, bf16=False,  # Pascal / Turing: no bf16
                             report_to=[], remove_unused_columns=False, seed=C.SEED)
    ckpts = sorted(glob.glob(os.path.join(a.out, "checkpoint-*")), key=lambda p: int(p.rsplit("-", 1)[-1]))
    if os.path.exists(done_flag):
        print("adapter weights found; only the held-out check is left")
    else:
        if ckpts:
            print("resuming from", ckpts[-1])
        Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collate).train(
            resume_from_checkpoint=ckpts[-1] if ckpts else None)
        model.save_pretrained(a.out)
    tok.save_pretrained(a.out)
    print("adapter saved ->", a.out)

    # held-out label accuracy with the exact inference scorer
    if dev:
        del model
        torch.cuda.empty_cache()
        from verifier import HFVerifier
        v = HFVerifier(base=a.base, adapter=a.out, load_4bit=not a.no_4bit)
        ok, per = 0, {}
        for r in dev:
            labels = C.LABELS_CHAIN if r["target"] in C.LABELS_CHAIN else C.LABELS_ENTAIL
            hit = v.score(r["messages"], labels)["label"] == r["target"]
            ok += hit
            n_hit = per.setdefault(r["target"], [0, 0])
            n_hit[0] += 1
            n_hit[1] += hit
        print(f"held-out accuracy {ok / len(dev):.3f} on {len(dev)} examples")
        for label, (n, h) in sorted(per.items()):  # accuracy alone hides a majority-class verifier
            print(f"  {label}: {h / n:.3f} on {n}")
        print(f"  balanced accuracy {sum(h / n for n, h in per.values()) / len(per):.3f}")
    res = {"train_examples": len(train), "heldout_examples": len(dev),
           "label_counts": {l: sum(r["target"] == l for r in rows) for l in sorted({r["target"] for r in rows})}}
    if dev:
        res.update(heldout_accuracy=round(ok / len(dev), 4),
                   per_label={l: {"n": n, "accuracy": round(h / n, 4)} for l, (n, h) in sorted(per.items())},
                   balanced_accuracy=round(sum(h / n for n, h in per.values()) / len(per), 4))
    with open(os.path.join(a.out, "heldout_accuracy.json"), "w") as f:
        json.dump(res, f, indent=1)
    print("saved", os.path.join(a.out, "heldout_accuracy.json"))


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/tokenizers/rq2_evidence_chain/scripts/train_verifier_r.py


In [6]:
%%writefile {SCRIPTS}/signoff.py
# block 12: simulated doctor sign-off
"""Block 12 (simulated doctor sign-off): turn the RQ2 run on the Verify-and-Ask notes into final notes.

For every case it combines
  * the verified S/O and the open questions from RQ1   (verify_and_ask_pipeline/.../output/cases/<case>.json)
  * each A/P statement's RQ2 record                    (runs/<split>/<run>.jsonl, written by run_rq2.py)
  * the flag threshold tuned on dev                    (reports/thresholds.json, written by evaluate_rq2.py --tune)
and applies the sign-off rule that stands in for the doctor:
  OK lines are kept; flagged lines are held back for doctor review (never deleted, never rewritten);
  open questions are passed on as pending items.

Only the REAL copy (the note's own A/P statements) is signed. The TEST copy (real + planted statements) is used
only for the "perfect doctor" numbers in signoff_summary.json: how many planted errors would still reach the
signed note (those RQ2 did not flag) and how many flags the doctor has to review.

Usage (same RQ2_* folders as the runs):
  python signoff.py --run U4_preap_ref3_link-llm_hf
Writes to ../rq2_on_verify_and_ask/ by default:
  doctor_notes/<case>.md    review version: S/O + every A/P line marked OK / FLAGGED with its evidence
  signed_notes/<case>.md    signed version: S/O + OK A/P lines only (reader 1; input to the patient summary)
  cases/<case>.json         everything about the case in one file
  index.csv                 one row per case
  signoff_summary.json      counts per split, incl. the perfect-doctor numbers
"""
import argparse
import csv
import json
from pathlib import Path

import config as C
from common import load_encounters, load_statements, read_json, read_jsonl, write_json

SPLITS = ("va_dev", "va_test", "va_trainorig")
SCORED = ("va_dev", "va_test")


def load_run(split, run):
    p = C.RUNS_DIR / split / f"{run}.jsonl"
    return {r["statement_id"]: r for r in read_jsonl(p)} if p.exists() else {}


def load_features(enc_id):
    p = C.RUNS_DIR / "features" / "preap" / f"{enc_id}.json"
    return {f["fid"]: f for f in read_json(p)["features"]} if p.exists() else {}


def judge(stmt, rec, tau):
    """-> status, score, evidence."""
    if rec is None or "p_reject" not in rec:
        return "not_checked", None, []
    status = "flagged" if rec["p_reject"] >= tau else "ok"
    return status, round(rec["p_reject"], 4), rec.get("components", [])


def evidence_lines(components, feats, status, tau):
    out = []
    for c in components:
        mark = " ← reason for flag" if status == "flagged" and c.get("p_broken", 0) >= tau else ""
        out.append(f"    - component: “{c['component']}” — P(chain broken) = {c.get('p_broken', 0):.2f}{mark}")
        for fid in c.get("feature_ids", []):
            f = feats.get(fid, {})
            turns = ", ".join(f.get("turns") or []) or "no turn cited"
            out.append(f"        - [{fid}] {f.get('text', '(feature not found)')} ({turns})")
        if not c.get("feature_ids"):
            out.append("        - no patient facts were linked")
        for p in c.get("passages", []):
            out.append(f"        - reference: MedlinePlus “{p.get('title', '')}” ({p.get('section', '')})")
    return out


def bullet_block(text):
    lines = [l.strip() for l in (text or "").splitlines() if l.strip()]
    return "\n".join(l if l.startswith(("*", "-", "•")) else f"* {l}" for l in lines) or "* (nothing documented)"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", required=True, help="RQ2 run to sign off, e.g. U4_preap_ref3_link-llm_hf")
    ap.add_argument("--threshold", type=float, help="override the dev-tuned threshold")
    ap.add_argument("--va-cases", default=str(C.REPO / "verify_and_ask_pipeline" / "Stage23_pipeline" / "output"
                                              / "cases"))
    ap.add_argument("--out", default=str(C.REPO / "rq2_on_verify_and_ask"))
    a = ap.parse_args()

    thr_file = C.REPORT_DIR / "thresholds.json"
    tau = a.threshold
    if tau is None:
        thr = read_json(thr_file) if thr_file.exists() else {}
        if a.run not in thr:
            raise SystemExit(f"No dev-tuned threshold for {a.run} in {thr_file}. "
                             f"Run: python evaluate_rq2.py --split va_dev --tune  (or pass --threshold)")
        tau = thr[a.run]
    print(f"run {a.run}  threshold {tau:.4f}")

    out = Path(a.out)
    for d in ("doctor_notes", "signed_notes", "cases"):
        (out / d).mkdir(parents=True, exist_ok=True)
    va_dir = Path(a.va_cases)
    index, summary = [], {"run": a.run, "threshold": tau, "rule": "flagged = P(chain broken) >= threshold; "
                          "flagged lines held back for doctor review, OK lines signed", "splits": {}}

    for split in SPLITS:
        if not (C.DATA_DIR / f"encounters_{split}.jsonl").exists():
            continue
        encs = load_encounters(split)
        real = [s for s in load_statements(split, include_planted=False)]
        test_copy = load_statements(split, include_planted=True) if split in SCORED else real
        recs = load_run(split, a.run)
        if not recs:
            print(f"[{split}] no run file runs/{split}/{a.run}.jsonl - skipped")
            continue

        # ---------------- perfect-doctor numbers on the test copy (known answers)
        if split in SCORED:
            valid = [s for s in test_copy if s["gold_label"] == "valid" and s["statement_id"] in recs]
            planted = [s for s in test_copy if s["gold_label"] == "unsupported" and s["statement_id"] in recs]
            flag = {sid: r["p_reject"] >= tau for sid, r in recs.items() if "p_reject" in r}
            n_enc = len({s["encounter_id"] for s in valid + planted}) or 1
            reach = sum(not flag.get(s["statement_id"], False) for s in planted)
            summary["splits"][split] = dict(
                encounters=n_enc, valid_statements=len(valid), planted_statements=len(planted),
                valid_held_back=sum(flag.get(s["statement_id"], False) for s in valid),
                planted_flagged=len(planted) - reach,
                planted_reaching_signed_note=reach,
                planted_reaching_rate=round(reach / len(planted), 4) if planted else None,
                flags_per_note_test_copy=round(sum(flag.get(s["statement_id"], False)
                                                   for s in valid + planted) / n_enc, 3))

        # ---------------- real copy: one doctor note + one signed note per case
        by_enc = {}
        for s in real:
            by_enc.setdefault(s["encounter_id"], []).append(s)
        for enc_id, enc in sorted(encs.items()):
            case = json.loads((va_dir / f"{enc_id}.json").read_text(encoding="utf-8"))
            so = case["stage3_so"]
            gaps = [g for g in (case.get("gaps") or {}).get("gaps", [])]
            feats = load_features(enc_id)
            lines = []
            for s in by_enc.get(enc_id, []):
                status, score, comps = judge(s, recs.get(s["statement_id"]), tau)
                lines.append(dict(statement_id=s["statement_id"], section=s["section"], text=s["claim"],
                                  status=status, p_chain_broken=score, components=comps))
            held = [l for l in lines if l["status"] != "ok"]

            # doctor-facing review note
            md = [f"# {enc_id} — note for doctor review", "",
                  f"RQ2 run `{a.run}`, flag threshold {tau:.2f}. ✓ = facts justify the line; "
                  f"⚑ = flagged for review (kept in this view, held back from the signed note).", "",
                  "## Subjective (verified by RQ1)", bullet_block(so.get("subjective")), "",
                  "## Objective (verified by RQ1)", bullet_block(so.get("objective")), ""]
            for sect in ("Assessment", "Plan"):
                md.append(f"## {sect} (checked by RQ2)")
                sl = [l for l in lines if l["section"] == sect]
                if not sl:
                    md.append("* (no statements)")
                for l in sl:
                    tag = {"ok": "✓", "flagged": "⚑ FLAGGED", "not_checked": "⚑ NOT CHECKED"}[l["status"]]
                    sc = f" — P(chain broken) = {l['p_chain_broken']:.2f}" if l["p_chain_broken"] is not None else ""
                    md.append(f"* {tag} {l['text']}{sc}")
                    md.extend(evidence_lines(l["components"], feats, l["status"], tau))
                md.append("")
            md.append("## Open questions from RQ1 (not answered)")
            md.extend([f"* [{g.get('category', '')}] {g.get('question', '')}" for g in gaps] or ["* none"])
            (out / "doctor_notes" / f"{enc_id}.md").write_text("\n".join(md) + "\n", encoding="utf-8")

            # signed note (reader 1)
            signed = {"subjective": so.get("subjective", ""), "objective": so.get("objective", ""),
                      "assessment": [l["text"] for l in lines if l["section"] == "Assessment" and l["status"] == "ok"],
                      "plan": [l["text"] for l in lines if l["section"] == "Plan" and l["status"] == "ok"]}
            sm = [f"# {enc_id} — signed SOAP note", "",
                  "## Subjective", bullet_block(signed["subjective"]), "",
                  "## Objective", bullet_block(signed["objective"]), "",
                  "## Assessment", "\n".join(f"* {t}" for t in signed["assessment"]) or "* (none signed)", "",
                  "## Plan", "\n".join(f"* {t}" for t in signed["plan"]) or "* (none signed)", "",
                  f"_{len(held)} A/P line(s) held back for doctor review; {len(gaps)} open question(s) pending._"]
            (out / "signed_notes" / f"{enc_id}.md").write_text("\n".join(sm) + "\n", encoding="utf-8")

            write_json(out / "cases" / f"{enc_id}.json", dict(
                case_id=enc_id, rq2_split=split, rq1_split=enc.get("rq1_split"),
                aci_source_split=enc.get("aci_source_split"), scored=split in SCORED,
                rq2_run=a.run, threshold=tau, so=so, open_questions=gaps, ap_lines=lines,
                signed_note=signed, conversation_turns=enc["turns"]))
            index.append(dict(case_id=enc_id, rq2_split=split, rq1_split=enc.get("rq1_split"),
                              aci_source_split=enc.get("aci_source_split"), scored=split in SCORED,
                              n_assessment=sum(l["section"] == "Assessment" for l in lines),
                              n_plan=sum(l["section"] == "Plan" for l in lines),
                              n_flagged=sum(l["status"] == "flagged" for l in lines),
                              n_not_checked=sum(l["status"] == "not_checked" for l in lines),
                              n_signed_ap=len(signed["assessment"]) + len(signed["plan"]),
                              n_open_questions=len(gaps),
                              n_open_subjective=sum(g.get("category") == "subjective" for g in gaps),
                              n_open_objective=sum(g.get("category") == "objective" for g in gaps)))
        done = [i for i in index if i["rq2_split"] == split]
        summary["splits"].setdefault(split, {}).update(
            cases=len(done), real_ap_lines=sum(i["n_assessment"] + i["n_plan"] for i in done),
            real_flagged=sum(i["n_flagged"] for i in done), real_not_checked=sum(i["n_not_checked"] for i in done),
            flags_per_note_real=round(sum(i["n_flagged"] for i in done) / max(len(done), 1), 3))
        print(f"[{split}] {len(done)} cases signed")

    with open(out / "index.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(index[0].keys()))
        w.writeheader()
        w.writerows(index)
    write_json(out / "signoff_summary.json", summary)
    print(json.dumps(summary, indent=1))
    print(f"wrote {out}/")


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/tokenizers/rq2_evidence_chain/scripts/signoff.py


In [7]:
%%writefile {SCRIPTS}/patient_summary.py
# blocks 13-14: checked patient summary + H3
"""Blocks 13-14: plain-language patient summary from the signed note, each sentence checked, plus H3.

13 Summary writer  (generator, Qwen3-14B): rewrites the SIGNED note (signoff.py) for the patient.
14 Summary checker (verifier 3B, S/O entailment prompt): each summary sentence must be supported by the signed
                    note (P(supported) >= --tau, default 0.5); unsupported sentences are dropped and logged.

H3 control: the same writer on the UNVERIFIED record (RQ1 Stage 1 S/O + Stage 1 A/P, nothing checked), with no
checker. Both summaries are then measured the same way:
  unsupported_vs_signed   sentences not supported by the signed note (content the verified record does not hold)
  unsupported_vs_dialogue sentences not supported by the conversation (independent check)
  next_step_recall        share of signed Plan lines that the summary covers
  fk_grade                Flesch-Kincaid grade (reading level; lower = easier)
Warning-sign recall is not computed: the notes rarely state warning signs, so there is nothing to recall.

Usage (after signoff.py, generator server and verifier as for run_rq2.py):
  python patient_summary.py                     # all cases in ../rq2_on_verify_and_ask/cases
  python patient_summary.py --limit 5           # quick check
Writes, next to signoff.py's output:
  patient_summaries/<case>.md             final verified summary (reader 2)
  patient_summaries_unverified/<case>.md  H3 control summary
  summaries/<case>.json                   both summaries, every sentence with its check scores
  patient_summary_metrics.json / .md      H3 table (va_test, and all cases)
Resumable: finished cases are skipped.
"""
import argparse
import json
import re
from pathlib import Path

import config as C
from common import read_json, split_sentences, strip_bullet, write_json
from prompts import entail_messages

SYSTEM = ("You write plain-language visit summaries for patients, based only on the clinician's note you are "
          "given. Output JSON only.")
USER = """Clinician's note:
{note}

Write a short summary of this visit for the patient.
Rules:
- Use ONLY information in the note. Do not add any diagnosis, test, medicine, dose, number, timing or advice
  that the note does not contain. If the note has no plan, say that the next steps will be discussed.
- Speak to the patient as "you". Short sentences, everyday words, about a grade 6-8 reading level.
  If you must use a medical word, explain it in a few plain words.
- 5 to 10 sentences, grouped in these parts (skip a part if the note has nothing for it):
  "why_you_came", "what_we_found", "what_it_may_be", "next_steps".
Return JSON: {{"sentences": [{{"part": "why_you_came", "text": "..."}}, ...]}}"""
PART_TITLES = {"why_you_came": "Why you came in", "what_we_found": "What we found",
               "what_it_may_be": "What it may be", "next_steps": "What happens next"}


def note_text(so, assessment, plan):
    def block(x):
        if isinstance(x, list):
            return "\n".join(f"- {t}" for t in x) or "- (none)"
        lines = [strip_bullet(l.strip()) for l in (x or "").splitlines() if l.strip()]
        return "\n".join(f"- {l}" for l in lines) or "- (none)"
    return (f"SUBJECTIVE\n{block(so.get('subjective'))}\n\nOBJECTIVE\n{block(so.get('objective'))}\n\n"
            f"ASSESSMENT\n{block(assessment)}\n\nPLAN\n{block(plan)}")


def syllables(word):
    w = re.sub(r"[^a-z]", "", word.lower())
    if not w:
        return 0
    groups = re.findall(r"[aeiouy]+", w)
    n = len(groups) - (1 if w.endswith("e") and len(groups) > 1 and not w.endswith("le") else 0)
    return max(1, n)


def fk_grade(sentences):
    words = [w for s in sentences for w in re.findall(r"[A-Za-z']+", s)]
    if not sentences or not words:
        return None
    return round(0.39 * len(words) / len(sentences) + 11.8 * sum(map(syllables, words)) / len(words) - 15.59, 2)


def p_supported(verifier, evidence, claim):
    return verifier.score(entail_messages(evidence, claim), C.LABELS_ENTAIL)["probs"]["supported"]


def write_summary(llm, note):
    out = llm.chat_json(SYSTEM, USER.format(note=note), temperature=0.0, max_tokens=900) or {}
    sents = []
    for item in out.get("sentences", []) if isinstance(out, dict) else []:
        txt = (item.get("text") or "").strip() if isinstance(item, dict) else ""
        for s in split_sentences(txt):
            sents.append(dict(part=item.get("part", "next_steps"), text=s))
    return sents


def stub_writer(system, user):
    """Offline test only: copies the note's lines as 'summary' sentences."""
    lines = [l[2:] for l in user.split("Write a short summary")[0].splitlines() if l.startswith("- ")][:8]
    return json.dumps({"sentences": [{"part": "what_we_found", "text": l} for l in lines if l != "(none)"]})


def measure(verifier, sents, signed, dialogue, plan_lines, tau):
    for s in sents:
        s.setdefault("p_vs_signed", round(p_supported(verifier, signed, s["text"]), 4))
        s["p_vs_dialogue"] = round(p_supported(verifier, dialogue, s["text"]), 4)
    summary_txt = " ".join(s["text"] for s in sents)
    covered = [p_supported(verifier, summary_txt, p) >= tau for p in plan_lines] if sents else [False] * len(plan_lines)
    return dict(n_sentences=len(sents),
                unsupported_vs_signed=sum(s["p_vs_signed"] < tau for s in sents),
                unsupported_vs_dialogue=sum(s["p_vs_dialogue"] < tau for s in sents),
                next_step_recall=round(sum(covered) / len(plan_lines), 4) if plan_lines else None,
                fk_grade=fk_grade([s["text"] for s in sents]))


def to_md(case_id, sents, title):
    md = [f"# {title} — {case_id}", ""]
    for part, head in PART_TITLES.items():
        ps = [s["text"] for s in sents if s["part"] == part]
        if ps:
            md += [f"## {head}", " ".join(ps), ""]
    if len(md) == 2:
        md.append("_No summary sentences passed the check._")
    return "\n".join(md) + "\n"


def avg(xs):
    xs = [x for x in xs if x is not None]
    return round(sum(xs) / len(xs), 4) if xs else None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--final", default=str(C.REPO / "rq2_on_verify_and_ask"), help="signoff.py output folder")
    ap.add_argument("--va-cases", default=str(C.REPO / "verify_and_ask_pipeline" / "Stage23_pipeline" / "output"
                                              / "cases"))
    ap.add_argument("--tau", type=float, default=0.5, help="P(supported) needed to keep a summary sentence")
    ap.add_argument("--limit", type=int)
    a = ap.parse_args()

    from llm import GenLLM
    from verifier import make_verifier
    llm, verifier = GenLLM(), make_verifier()
    if C.GEN_BACKEND == "stub":
        llm.stub_fn = stub_writer
    final = Path(a.final)
    for d in ("patient_summaries", "patient_summaries_unverified", "summaries"):
        (final / d).mkdir(parents=True, exist_ok=True)

    cases = sorted((final / "cases").glob("*.json"))[:a.limit]
    rows = []
    for n, p in enumerate(cases, 1):
        case = read_json(p)
        cid = case["case_id"]
        out_p = final / "summaries" / f"{cid}.json"
        if out_p.exists():
            rows.append(read_json(out_p))
            continue
        signed = case["signed_note"]
        signed_txt = note_text(signed, signed["assessment"], signed["plan"])
        dialogue = "\n".join(f"[{t['tid']}] {t['speaker']}: {t['text']}" for t in case["conversation_turns"])
        va = json.loads((Path(a.va_cases) / f"{cid}.json").read_text(encoding="utf-8"))
        unverified_txt = note_text(va["stage1_so"], va["stage1_ap"].get("assessment"), va["stage1_ap"].get("plan"))

        # 13 + 14: verified summary
        draft = write_summary(llm, signed_txt)
        for s in draft:
            s["p_vs_signed"] = round(p_supported(verifier, signed_txt, s["text"]), 4)
            s["kept"] = s["p_vs_signed"] >= a.tau
        final_sents = [dict(s) for s in draft if s["kept"]]
        # H3 control: unverified record, no checker
        control = write_summary(llm, unverified_txt)

        m_final = measure(verifier, final_sents, signed_txt, dialogue, signed["plan"], a.tau)
        m_ctrl = measure(verifier, control, signed_txt, dialogue, signed["plan"], a.tau)
        row = dict(case_id=cid, rq2_split=case["rq2_split"], scored=case["scored"],
                   draft=draft, dropped=[s for s in draft if not s["kept"]],
                   verified_summary=final_sents, unverified_summary=control,
                   metrics_verified=m_final, metrics_unverified=m_ctrl,
                   draft_unsupported_vs_signed=sum(not s["kept"] for s in draft))
        write_json(out_p, row)
        (final / "patient_summaries" / f"{cid}.md").write_text(
            to_md(cid, final_sents, "Your visit summary"), encoding="utf-8")
        (final / "patient_summaries_unverified" / f"{cid}.md").write_text(
            to_md(cid, control, "Visit summary from the UNVERIFIED note (H3 control)"), encoding="utf-8")
        rows.append(row)
        print(f"  [{n}/{len(cases)}] {cid}: kept {len(final_sents)}/{len(draft)} sentences", flush=True)

    def table(sel):
        out = {}
        for key, name in (("metrics_verified", "verified (signed note + checker)"),
                          ("metrics_unverified", "unverified record, no checker")):
            ms = [r[key] for r in sel]
            out[name] = dict(cases=len(ms),
                             unsupported_vs_signed_per_summary=avg([m["unsupported_vs_signed"] for m in ms]),
                             unsupported_vs_dialogue_per_summary=avg([m["unsupported_vs_dialogue"] for m in ms]),
                             next_step_recall=avg([m["next_step_recall"] for m in ms]),
                             fk_grade=avg([m["fk_grade"] for m in ms]),
                             sentences_per_summary=avg([m["n_sentences"] for m in ms]))
        out["checker"] = dict(draft_sentences_dropped_per_summary=avg([r["draft_unsupported_vs_signed"] for r in sel]))
        return out

    res = {"tau": a.tau, "va_test": table([r for r in rows if r["rq2_split"] == "va_test"]),
           "all_cases": table(rows)}
    write_json(final / "patient_summary_metrics.json", res)
    md = ["# H3: patient summaries", "",
          "Unsupported = P(supported) below the threshold. Lower is better for unsupported counts and reading grade.", ""]
    for scope in ("va_test", "all_cases"):
        t = res[scope]
        md += [f"## {scope}", "", "| summary built from | cases | unsupported vs signed note | unsupported vs dialogue "
               "| next-step recall | reading grade (FK) | sentences |", "|---|---|---|---|---|---|---|"]
        for name, v in t.items():
            if name == "checker":
                continue
            md.append(f"| {name} | {v['cases']} | {v['unsupported_vs_signed_per_summary']} | "
                      f"{v['unsupported_vs_dialogue_per_summary']} | {v['next_step_recall']} | {v['fk_grade']} | "
                      f"{v['sentences_per_summary']} |")
        md += ["", f"Checker dropped {t['checker']['draft_sentences_dropped_per_summary']} draft sentence(s) "
                   f"per summary.", ""]
    (final / "patient_summary_metrics.md").write_text("\n".join(md), encoding="utf-8")
    print("\n".join(md))


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/tokenizers/rq2_evidence_chain/scripts/patient_summary.py


In [8]:
%%writefile {SCRIPTS}/export_results.py
# collect every block's results into CSV / JSON
"""Collect everything RQ2 produced on the Verify-and-Ask notes into a few CSV / JSON files.

rq2_results/
  blocks/b04_statements.csv      one row per A/P statement (real + planted; filler lines marked excluded)
  blocks/b05_components.csv      one row per piece after the claim splitter
  blocks/b06_facts.csv           one row per patient fact found by the fact collector
  blocks/b07_fact_checks.csv     one row per fact: P(supported) in its cited turns, kept or dropped
  blocks/b08_links.csv           one row per piece: the facts the linker picked
  blocks/b09_references.csv      one row per piece (U4): MedlinePlus passages and their closeness
  blocks/b10_chain_scores.csv    one row per piece: P(chain broken) without (U3) and with (U4) passages
  blocks/b11_flags.csv           one row per statement: score and flag for every condition (U1..U4)
  blocks/b12_signoff.csv         one row per case: A/P lines, flagged, signed, open questions
  blocks/b13_summary_drafts.csv  one row per summary sentence as written (verified-note draft + H3 control)
  blocks/b14_summary_checks.csv  one row per sentence: checked against the signed note and the conversation
  metrics.json                   every number, grouped by block
  combined.json                  one entry per case with everything joined

Only the scored splits (va_dev, va_test) are exported.
  python export_results.py --final ../rq2_on_verify_and_ask --out ../rq2_results [--train-info <json>]
"""
import argparse
import csv
import json
from collections import Counter, defaultdict
from pathlib import Path
from statistics import mean

import config as C
from common import load_encounters, load_statements, read_json, read_jsonl

SPLITS = ("va_dev", "va_test")
CONDS = ("U1", "U2", "U3", "U4")


def write_csv(path, rows, cols=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    cols = cols or (list(rows[0].keys()) if rows else ["empty"])
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)
    return len(rows)


def avg(xs):
    xs = [x for x in xs if x is not None]
    return round(mean(xs), 4) if xs else None


def share(n, d):
    return round(n / d, 4) if d else None


def find_runs(split):
    """condition -> run name, primary runs only (whole dialogue, LLM linker, pre-A/P)."""
    out = {}
    for p in sorted((C.RUNS_DIR / split).glob("U*_preap*.jsonl")):
        name = p.stem
        cond = name[:2]
        if "_top" in name or "link-retrieval" in name or cond not in CONDS:
            continue
        out.setdefault(cond, name)
    return out


def read_metrics_csv(path):
    if not path.exists():
        return []
    rows = []
    for r in csv.DictReader(open(path, encoding="utf-8")):
        rows.append({k: (float(v) if v not in ("", None) and k not in ("run", "section", "error_type", "retention_ci")
                         else v) for k, v in r.items()})
    return rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--final", default=str(C.REPO / "rq2_on_verify_and_ask"))
    ap.add_argument("--out", default=str(C.REPO / "rq2_results"))
    ap.add_argument("--train-info", help="JSON with verifier training info (written by the notebook)")
    a = ap.parse_args()
    final, out = Path(a.final), Path(a.out)
    blocks = out / "blocks"
    thr = read_json(C.REPORT_DIR / "thresholds.json") if (C.REPORT_DIR / "thresholds.json").exists() else {}

    rows = {k: [] for k in ("b04", "b05", "b06", "b07", "b08", "b09", "b10", "b11", "b12", "b13", "b14")}
    combined = {}
    runs_used = {}
    for split in SPLITS:
        if not (C.DATA_DIR / f"encounters_{split}.jsonl").exists():
            continue
        encs = load_encounters(split)
        runs = find_runs(split)
        runs_used[split] = runs
        recs = {c: {r["statement_id"]: r for r in read_jsonl(C.RUNS_DIR / split / f"{n}.jsonl")}
                for c, n in runs.items()}
        stmts = load_statements(split, include_planted=True)
        all_real = read_jsonl(C.DATA_DIR / f"statements_{split}.jsonl")   # incl. excluded filler lines
        comps = {r["statement_id"]: r["components"] for r in read_jsonl(C.RUNS_DIR / split / "chain_components.jsonl")} \
            if (C.RUNS_DIR / split / "chain_components.jsonl").exists() else {}
        links = {r["statement_id"]: r["feature_ids"] for r in read_jsonl(C.RUNS_DIR / split / "chain_links_preap.jsonl")} \
            if (C.RUNS_DIR / split / "chain_links_preap.jsonl").exists() else {}

        # ---- block 4: statements (real incl. excluded filler + planted)
        kept_ids = {s["statement_id"] for s in stmts}
        for s in all_real + [s for s in stmts if s.get("origin") == "planted"]:
            rows["b04"].append(dict(statement_id=s["statement_id"], case_id=s["encounter_id"], split=split,
                                    section=s["section"], stmt_type=s.get("stmt_type", ""),
                                    origin="planted" if s.get("origin") == "planted" else "real",
                                    gold_label=s["gold_label"], error_type=s.get("error_type", ""),
                                    excluded=int(s["statement_id"] not in kept_ids), text=s["claim"]))
        # ---- blocks 6-7: facts
        for eid in encs:
            p = C.RUNS_DIR / "features" / "preap" / f"{eid}.json"
            feats = read_json(p)["features"] if p.exists() else []
            for f in feats:
                rows["b06"].append(dict(case_id=eid, split=split, fact_id=f["fid"], category=f.get("category", ""),
                                        turns=",".join(f.get("turns") or []), text=f["text"]))
                rows["b07"].append(dict(case_id=eid, split=split, fact_id=f["fid"],
                                        p_supported=f.get("ground_p"), kept=int(bool(f.get("grounded", True)))))
        # ---- blocks 5, 8, 9, 10, 11 per statement
        for s in stmts:
            sid, eid = s["statement_id"], s["encounter_id"]
            cl = comps.get(sid, [])
            u3 = (recs.get("U3", {}).get(sid) or {}).get("components", [])
            u4 = (recs.get("U4", {}).get(sid) or {}).get("components", [])
            for i, c in enumerate(cl):
                rows["b05"].append(dict(statement_id=sid, case_id=eid, split=split, component_index=i,
                                        n_components=len(cl), component=c))
                ids = links.get(f"{sid}#{i}#llm", [])
                rows["b08"].append(dict(statement_id=sid, case_id=eid, split=split, component_index=i,
                                        n_linked=len(ids), linked_fact_ids=",".join(ids)))
                c4 = u4[i] if i < len(u4) else {}
                ps = c4.get("passages", [])
                if c4:
                    rows["b09"].append(dict(statement_id=sid, case_id=eid, split=split, component_index=i,
                                            reference_found=int(bool(c4.get("reference_found"))),
                                            n_passages=len(ps), titles=" | ".join(p.get("title", "") for p in ps),
                                            sections=" | ".join(p.get("section", "") for p in ps),
                                            closeness=" | ".join(str(p.get("cos", "")) for p in ps)))
                c3 = u3[i] if i < len(u3) else {}
                rows["b10"].append(dict(statement_id=sid, case_id=eid, split=split, component_index=i,
                                        gold_label=s["gold_label"], p_broken_U3=c3.get("p_broken"),
                                        p_broken_U4=c4.get("p_broken")))
            flag_row = dict(statement_id=sid, case_id=eid, split=split, section=s["section"],
                            origin="planted" if s.get("origin") == "planted" else "real",
                            gold_label=s["gold_label"], error_type=s.get("error_type", ""), text=s["claim"])
            for c in CONDS:
                r = recs.get(c, {}).get(sid)
                run = runs.get(c)
                if not r or "p_reject" not in r:
                    flag_row[f"score_{c}"] = flag_row[f"flag_{c}"] = None
                    continue
                flag_row[f"score_{c}"] = round(r["p_reject"], 4)
                if c == "U1":   # published uMedSum decision (thresholds from the paper)
                    flag_row["flag_U1"] = int(bool(r["reject"]))
                else:
                    flag_row[f"flag_{c}"] = int(r["p_reject"] >= thr[run]) if run in thr else None
                flag_row[f"seconds_{c}"] = r.get("seconds")
            rows["b11"].append(flag_row)

    # ---- blocks 12-14 from signoff.py / patient_summary.py
    for p in sorted((final / "cases").glob("*.json")):
        case = read_json(p)
        if case.get("rq2_split") not in SPLITS:
            continue
        lines = case["ap_lines"]
        rows["b12"].append(dict(case_id=case["case_id"], split=case["rq2_split"],
                                n_ap_lines=len(lines), n_assessment=sum(l["section"] == "Assessment" for l in lines),
                                n_plan=sum(l["section"] == "Plan" for l in lines),
                                n_flagged=sum(l["status"] == "flagged" for l in lines),
                                n_not_checked=sum(l["status"] == "not_checked" for l in lines),
                                n_signed=len(case["signed_note"]["assessment"]) + len(case["signed_note"]["plan"]),
                                n_open_questions=len(case["open_questions"])))
        sp = final / "summaries" / f"{case['case_id']}.json"
        summ = read_json(sp) if sp.exists() else None
        if summ:
            for variant, sents in (("verified_draft", summ["draft"]), ("unverified", summ["unverified_summary"])):
                for i, s in enumerate(sents):
                    rows["b13"].append(dict(case_id=case["case_id"], split=case["rq2_split"], variant=variant,
                                            sentence_index=i, part=s.get("part", ""), text=s["text"]))
            fin = {s["text"] for s in summ["verified_summary"]}
            for i, s in enumerate(summ["draft"]):
                rows["b14"].append(dict(case_id=case["case_id"], split=case["rq2_split"], variant="verified",
                                        sentence_index=i, text=s["text"], p_vs_signed=s.get("p_vs_signed"),
                                        p_vs_dialogue=next((x.get("p_vs_dialogue") for x in summ["verified_summary"]
                                                            if x["text"] == s["text"]), None),
                                        kept=int(s["text"] in fin)))
            for i, s in enumerate(summ["unverified_summary"]):
                rows["b14"].append(dict(case_id=case["case_id"], split=case["rq2_split"], variant="unverified",
                                        sentence_index=i, text=s["text"], p_vs_signed=s.get("p_vs_signed"),
                                        p_vs_dialogue=s.get("p_vs_dialogue"), kept=""))
        combined[case["case_id"]] = dict(case, patient_summary=summ,
                                         facts=[r for r in rows["b06"] if r["case_id"] == case["case_id"]],
                                         scores=[r for r in rows["b11"] if r["case_id"] == case["case_id"]])

    names = dict(b04="b04_statements", b05="b05_components", b06="b06_facts", b07="b07_fact_checks",
                 b08="b08_links", b09="b09_references", b10="b10_chain_scores", b11="b11_flags",
                 b12="b12_signoff", b13="b13_summary_drafts", b14="b14_summary_checks")
    counts = {names[k]: write_csv(blocks / f"{names[k]}.csv", v) for k, v in rows.items()}

    # ---------------------------------------------------------------- metrics.json
    def by_split(rs):
        g = defaultdict(list)
        for r in rs:
            g[r["split"]].append(r)
        return g

    m = {"files": counts, "runs": runs_used, "thresholds": thr}
    real = [r for r in rows["b04"] if r["origin"] == "real"]
    m["b04_statement_splitter"] = {sp: dict(
        cases=len({r["case_id"] for r in rs}),
        statements_kept=sum(not r["excluded"] for r in rs if r["origin"] == "real"),
        filler_lines_excluded=sum(r["excluded"] for r in rs if r["origin"] == "real"),
        assessment=sum(r["section"] == "Assessment" and not r["excluded"] for r in rs if r["origin"] == "real"),
        plan=sum(r["section"] == "Plan" and not r["excluded"] for r in rs if r["origin"] == "real"),
        statements_per_case=share(sum(not r["excluded"] for r in rs if r["origin"] == "real"),
                                  len({r["case_id"] for r in rs})),
        planted=dict(Counter(r["error_type"] for r in rs if r["origin"] == "planted")))
        for sp, rs in by_split(rows["b04"]).items()}
    m["b05_claim_splitter"] = {sp: dict(
        pieces=len(rs), statements=len({r["statement_id"] for r in rs}),
        share_split=share(len({r["statement_id"] for r in rs if r["n_components"] > 1}), len({r["statement_id"] for r in rs})),
        pieces_per_statement=share(len(rs), len({r["statement_id"] for r in rs})))
        for sp, rs in by_split(rows["b05"]).items()}
    m["b06_fact_collector"] = {sp: dict(
        facts=len(rs), facts_per_case=share(len(rs), len({r["case_id"] for r in rs})),
        by_category=dict(Counter(r["category"] for r in rs)),
        cases_without_facts=len({e for e in {r["case_id"] for r in rows["b04"] if r["split"] == sp}}
                                - {r["case_id"] for r in rs}))
        for sp, rs in by_split(rows["b06"]).items()}
    m["b07_fact_checker"] = {sp: dict(facts=len(rs), share_kept=share(sum(r["kept"] for r in rs), len(rs)),
                                      mean_p_supported=avg([r["p_supported"] for r in rs]))
                             for sp, rs in by_split(rows["b07"]).items()}
    m["b08_fact_linker"] = {sp: dict(pieces=len(rs), mean_facts_linked=avg([r["n_linked"] for r in rs]),
                                     share_pieces_without_facts=share(sum(r["n_linked"] == 0 for r in rs), len(rs)))
                            for sp, rs in by_split(rows["b08"]).items()}
    m["b09_reference_lookup"] = {sp: dict(pieces=len(rs),
                                          reference_coverage=share(sum(r["reference_found"] for r in rs), len(rs)),
                                          mean_passages=avg([r["n_passages"] for r in rs]))
                                 for sp, rs in by_split(rows["b09"]).items()}
    m["b10_chain_judge"] = {sp: {f"mean_p_broken_{c}_{g}": avg([r[f"p_broken_{c}"] for r in rs if r["gold_label"] == lab])
                                 for c in ("U3", "U4") for g, lab in (("real", "valid"), ("planted", "unsupported"))}
                            for sp, rs in by_split(rows["b10"]).items()}
    m["b11_flagger_rq2"] = {}
    for sp in SPLITS:
        rep = C.REPORT_DIR / sp
        if not rep.exists():
            continue
        m["b11_flagger_rq2"][sp] = dict(
            table=read_metrics_csv(rep / "metrics.csv"),
            recall_by_error_type=read_metrics_csv(rep / "recall_by_error_type.csv"),
            paired_comparisons_and_H2=read_json(rep / "summary.json") if (rep / "summary.json").exists() else None,
            note="test numbers are the results; dev numbers only chose the thresholds")
    m["b12_signoff"] = read_json(final / "signoff_summary.json") if (final / "signoff_summary.json").exists() else None
    m["b13_b14_patient_summary_H3"] = read_json(final / "patient_summary_metrics.json") \
        if (final / "patient_summary_metrics.json").exists() else None
    m["verifier_training"] = read_json(a.train_info) if a.train_info and Path(a.train_info).exists() else None

    out.mkdir(parents=True, exist_ok=True)
    (out / "metrics.json").write_text(json.dumps(m, indent=1, ensure_ascii=False), encoding="utf-8")
    (out / "combined.json").write_text(json.dumps(combined, indent=1, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(counts, indent=1))
    print(f"wrote {out}/ (blocks/*.csv, metrics.json, combined.json: {len(combined)} cases)")


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/tokenizers/rq2_evidence_chain/scripts/export_results.py


## 4 · Install packages (every session, about 5 minutes)
vLLM is installed first because it brings its own PyTorch. The `ERROR: pip's dependency resolver` warning that sometimes appears can be ignored.

In [9]:
!pip install -q vllm
!pip install -q -r {RQ2}/requirements.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.0 which is incompatible.


## 5 · Helpers
- `sh()` runs a command and stops the notebook if it fails.
- `start_generator()` and `stop_generator()` start and stop Qwen3-14B.
- `use_folders()` switches between the training folders and the verify-and-ask folders, so the two never mix. They're kept apart because saved fact lists are stored per case ID, and the same IDs exist in both.

In [10]:
import os, json, time, glob, subprocess
from pathlib import Path

def sh(cmd):
    """Run a shell command in the RQ2 folder, show its output live, stop the notebook if it fails."""
    p = subprocess.Popen(cmd, shell=True, cwd=RQ2, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=os.environ.copy())
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"command failed (exit {p.returncode}): {cmd}")

def generator_ready():
    return subprocess.run("curl -sf localhost:8000/health", shell=True).returncode == 0

def start_generator():
    """Start Qwen3-14B with vLLM in the background (first time: 5-10 min, mostly download)."""
    if generator_ready():
        print("generator already running"); return
    subprocess.Popen(
        f"nohup python -m vllm.entrypoints.openai.api_server --model {PROFILE['model']} "
        f"--served-model-name Qwen3-14B --max-model-len {PROFILE['max_len']} "
        f"--gpu-memory-utilization {PROFILE['util']} --reasoning-parser qwen3 --port 8000 "
        f"> {LOGS}/vllm.log 2>&1 &", shell=True)
    for _ in range(120):
        if generator_ready():
            print("generator READY"); return
        time.sleep(10)
    get_ipython().system(f"tail -n 30 {LOGS}/vllm.log")
    raise RuntimeError("generator did not start: see the log above")

def stop_generator():
    subprocess.run("pkill -f vllm.entrypoints", shell=True)
    time.sleep(15)
    print("generator stopped")

BASE_ENV = {
    "GEN_BACKEND": "openai", "GEN_BASE_URL": "http://127.0.0.1:8000/v1", "GEN_MODEL": "Qwen3-14B",
    "GEN_JSON_MODE": "1", "GEN_MAX_TOKENS": "2048",
    "VERIFIER_BACKEND": "hf", "VERIFIER_BASE_MODEL": "Qwen/Qwen2.5-3B-Instruct",
    "VERIFIER_ADAPTER": f"{WORK}/models/verifier_qlora", "VERIFIER_4BIT": "1", "VERIFIER_NAME": "hf",
    "U1_DECOMPOSER": "gen",                       # U1's sentence splitting uses Qwen3-14B (no LLaMA here)
    "RQ2_REF_DIR": f"{WORK}/reference_corpus",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "PYTHONPATH": RQ2,                            # lets the helper scripts import the RQ2 code
}
os.environ.update(BASE_ENV)

def use_folders(stage):
    """'aci_train' = ACI-Bench train (verifier training); 'va' = the verify-and-ask run."""
    os.environ.update({"RQ2_DATA_DIR": f"{WORK}/{stage}/data", "RQ2_RUNS_DIR": f"{WORK}/{stage}/runs",
                       "RQ2_REPORT_DIR": f"{WORK}/{stage}/reports"})
    print("folders ->", f"{WORK}/{stage}/")

if EXISTING_ADAPTER and not os.path.exists(f"{WORK}/models/verifier_qlora/adapter_model.safetensors"):
    sh(f'cp -r "{EXISTING_ADAPTER}" "{WORK}/models/verifier_qlora"')
print("ready")

ready


## 6 · Start the generator (Qwen3-14B)
Qwen3-14B does all the *writing* jobs: planting errors, listing facts (block 6), splitting statements (block 5), linking facts (block 8) and writing summaries (block 13). It never decides anything.

In [11]:
start_generator()

generator READY


## 7 · Smoke test (first session only, about 1 minute)
Runs the whole RQ2 code once with stand-in models and no GPU work, to catch setup problems early. It must end with `SMOKE TEST OK`.

In [12]:
marker = f"{WORK}/.smoke_ok"
if RUN_SMOKE_TEST and not os.path.exists(marker):
    sh(f"bash tests/smoke_test.sh > {LOGS}/smoke.log 2>&1; rc=$?; tail -n 3 {LOGS}/smoke.log; exit $rc")
    Path(marker).touch()
else:
    print("skipped")

reports -> /tmp/tmp.no90QQlRuD/reports/dev
0 rows -> /tmp/tmp.no90QQlRuD/data/steps_dev_U4_preap_ref3_link-llm_stub.csv
SMOKE TEST OK (/tmp/tmp.no90QQlRuD)


# Part A · Train the verifier

The verifier is trained **only on ACI-Bench train encounters** (67), never on the cases it's tested on. It learns from two kinds of practice examples:
- **A/P pairs** (block 10's question, *"do these facts justify this conclusion?"*): real diagnoses and plans with their patient facts → `chain_ok`; planted wrong ones and ones copied from another patient → `chain_broken`.
- **S/O pairs** (blocks 7 and 14 and U2's question, *"is this statement supported by this text?"*): real note sentences next to their dialogue → `supported`; edited copies (negation flipped, wrong side, dose or duration) → `contradicted`; sentences from another patient → `unsupported`.

All of Part A is skipped automatically once the trained verifier exists on Drive.

In [13]:
use_folders("aci_train")
ADAPTER = f"{WORK}/models/verifier_qlora"
TRAINED = os.path.exists(f"{ADAPTER}/adapter_model.safetensors") and os.path.exists(f"{ADAPTER}/heldout_accuracy.json")
TRAINED = TRAINED or (bool(EXISTING_ADAPTER) and os.path.exists(f"{ADAPTER}/adapter_model.safetensors"))
print("verifier already trained:", TRAINED)

folders -> /content/drive/MyDrive/tokenizers/rq2_evidence_chain/work/aci_train/
verifier already trained: False


### A1 · ACI-Bench train statements (1 minute)
Reads the 67 ACI-Bench train encounters and splits their reference A/P into statements.

In [14]:
if not TRAINED and not os.path.exists(f"{WORK}/aci_train/data/statements_train.jsonl"):
    sh("python build_dataset.py --splits train")
else:
    print("skipped")

[train] encounters=67 statements=790 (excluded 41)
   section: {'Assessment': 272, 'Plan': 518}
   type   : {'summary': 17, 'diagnosis': 118, 'reasoning': 137, 'test': 82, 'treatment': 180, 'education': 64, 'referral': 18, 'followup': 44, 'plan_other': 130}


### A2 · Plant wrong statements in train (about 15 minutes)
Qwen3-14B writes three wrong A/P statements per encounter: an unsupported diagnosis, a plan with no reason, and a conclusion a stated fact rules out. These become `chain_broken` examples. They're accepted without review because they're only for training.

In [15]:
if not TRAINED and not os.path.exists(f"{WORK}/aci_train/data/planted_train.jsonl"):
    sh("python plant_errors.py --split train")
    sh("python plant_errors.py --split train --merge --accept-all")
else:
    print("skipped")

  D2N001: 3 planted
  D2N002: 3 planted
  D2N003: 3 planted
  D2N004: 3 planted
  D2N005: 3 planted
  D2N006: 3 planted
  D2N007: 3 planted
  D2N008: 3 planted
  D2N009: 3 planted
  D2N010: 3 planted
  D2N011: 3 planted
  D2N012: 3 planted
  D2N013: 3 planted
  D2N014: 3 planted
  D2N015: 3 planted
  D2N016: 3 planted
  D2N017: 3 planted
  D2N018: 3 planted
  D2N019: 3 planted
  D2N020: 3 planted
  D2N021: 3 planted
  D2N022: 3 planted
  D2N023: 3 planted
  D2N024: 3 planted
  D2N025: 3 planted
  D2N026: 3 planted
  D2N027: 3 planted
  D2N028: 3 planted
  D2N029: 3 planted
  D2N030: 3 planted
  D2N031: 3 planted
  D2N032: 3 planted
  D2N033: 3 planted
  D2N034: 3 planted
  D2N035: 3 planted
  D2N036: 3 planted
  D2N037: 3 planted
  D2N038: 3 planted
  D2N039: 3 planted
  D2N040: 3 planted
  D2N041: 3 planted
  D2N042: 3 planted
  D2N043: 3 planted
  D2N044: 3 planted
  D2N045: 3 planted
  D2N046: 3 planted
  D2N047: 3 planted
  D2N048: 3 planted
  D2N049: 3 planted
  D2N050: 3 planted


### A3 · Remove the doctor's spoken plan from the train dialogues (about 30 minutes)
In ACI-Bench the doctor says the diagnosis and plan out loud. Those turns are removed so that training matches how the verifier is used: A/P checked against a conversation with no spoken conclusions. Encounters that fail are retried, up to three rounds.

In [16]:
def masked_ok():
    p = f"{WORK}/aci_train/data/encounters_train.jsonl"
    return os.path.exists(p) and all("turns_preap" in json.loads(l) for l in open(p))
if not TRAINED:
    for attempt in range(3):
        if masked_ok(): break
        sh("python mask_ap.py --split train")
    print("all train encounters masked:", masked_ok())
else:
    print("skipped")

  D2N001: 10 doctor turns edited/removed
  D2N002: 23 doctor turns edited/removed
  D2N003: 18 doctor turns edited/removed
  D2N004: 14 doctor turns edited/removed
  D2N005: 8 doctor turns edited/removed
  D2N006: 23 doctor turns edited/removed
  D2N007: 14 doctor turns edited/removed
  D2N008: 12 doctor turns edited/removed
  D2N009: 16 doctor turns edited/removed
  D2N010: 14 doctor turns edited/removed
  D2N011: 8 doctor turns edited/removed
  D2N012: 17 doctor turns edited/removed
  D2N013: 16 doctor turns edited/removed
  D2N014: 17 doctor turns edited/removed
  D2N015: 6 doctor turns edited/removed
  D2N016: 26 doctor turns edited/removed
  D2N017: 5 doctor turns edited/removed
  D2N018: 17 doctor turns edited/removed
  D2N019: 6 doctor turns edited/removed
  D2N020: 22 doctor turns edited/removed
  D2N021: 40 doctor turns edited/removed
  D2N022: 26 doctor turns edited/removed
  D2N023: 13 doctor turns edited/removed
  D2N024: 13 doctor turns edited/removed
  D2N025: 24 doctor t

### A4 · MedlinePlus reference library (about 5 minutes)
Downloads the free MedlinePlus health encyclopedia and cuts it into short passages. Block 9 searches this library. Needed even if training is skipped.

In [17]:
if not glob.glob(f"{WORK}/reference_corpus/chunks.jsonl"):
    sh("python medlineplus.py")
else:
    print("skipped: library already on Drive")

downloading https://medlineplus.gov/xml/mplus_topics_2026-11-13.xml
4000 passages from 1014 topics -> /content/drive/MyDrive/tokenizers/rq2_evidence_chain/work/reference_corpus/chunks.jsonl

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6370.56it/s]

Batches: 100%|██████████| 63/63 [00:02<00:00, 21.91it/s]
embeddings cached


### A5 · A/P practice examples (about 1.5–2 hours)
For every train encounter it runs blocks 6, 5 and 8 (facts, pieces, links) and saves each A/P statement with its facts as one practice example. Saved as it goes, so it resumes after a disconnect.

In [18]:
if not TRAINED and not os.path.exists(f"{WORK}/aci_train/data/chain_train.jsonl"):
    sh("FEATURE_GROUNDING=off python make_chain_training_data.py")
else:
    print("skipped")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5903.51it/s]
  D2N001: 29 examples
  D2N002: 23 examples
  D2N003: 20 examples
  D2N004: 25 examples
  D2N005: 11 examples
  D2N006: 24 examples
  D2N007: 11 examples
  D2N008: 26 examples
  D2N009: 17 examples
  D2N010: 7 examples
  D2N011: 17 examples
  D2N012: 20 examples
  D2N013: 16 examples
  D2N014: 19 examples
  D2N015: 23 examples
  D2N016: 24 examples
  D2N017: 11 examples
  D2N018: 12 examples
  D2N019: 12 examples
  D2N020: 16 examples
  D2N021: 13 examples
  D2N022: 20 examples
  D2N023: 16 examples
  D2N024: 12 examples
  WARNING D2N025: feature extraction returned nothing; will retry next run
  D2N026: 20 examples
  D2N027: 37 examples
  D2N028: 15 examples
  D2N029: 16 examples
  D2N030: 16 examples
  D2N031: 15 examples
  D2N032: 17 examples
  D2N033: 17 examples
  D2N034: 19 examples
  D2N035: 23 examples
  D2N036: 17 examples
  D2N037: 11 examples
  D2N038: 15 examples
  D2N039: 20 examples
  D2N040: 29 example

### A6 · S/O practice examples (seconds, no GPU)
Built by fixed rules from the ACI-Bench train reference notes.

In [19]:
if not TRAINED and not os.path.exists(f"{WORK}/aci_train/data/so_train.jsonl"):
    sh("python make_so_training_data.py")
else:
    print("skipped")

[train] 1922 pairs from 67 encounters -> /content/drive/MyDrive/tokenizers/rq2_evidence_chain/work/aci_train/data/so_train.jsonl
  label   : {'supported': 915, 'unsupported': 705, 'contradicted': 302}
  type    : {'none': 1218, 'fabricated_symptom': 402, 'negation_flip': 237, 'laterality_duration': 54, 'medication_dose': 11}
  evidence: {'full': 1316, 'window': 606}
  e.g. ('supported', 'none', 'full'): The patient sustained an injury to her right ankle when she slipped and fell on ice while taking her garbage out yesterday.
  e.g. ('unsupported', 'none', 'window'): Right knee: No pain with movement.
  e.g. ('unsupported', 'fabricated_symptom', 'full'): Range of Motion: Limited range of motion as well as pain on both flexion and extension of the knee.
  e.g. ('supported', 'none', 'window'): No masses, rebound or guarding.
  e.g. ('contradicted', 'negation_flip', 'full'): Constitutional: Denies low-grade fever.
  e.g. ('contradicted', 'laterality_duration', 'full'): The patient underwen

### A7 · Train the verifier (about 1–1.5 hours on A100, 2.5–3 hours on L4)
QLoRA training: the 3B base model stays frozen in 4-bit, and a small add-on is trained on both kinds of examples (2 passes, settings as in report §4.4). The generator is stopped during training to free GPU memory, then restarted.

**Resume:** a checkpoint is saved every 25 steps on Drive. After a disconnect, run all cells again and training continues from the newest checkpoint. At the end, the accuracy on held-out examples (10% of encounters, never trained on) is saved for the report.

In [20]:
if not TRAINED:
    stop_generator()
    sh(f"python {SCRIPTS}/train_verifier_r.py --data {WORK}/aci_train/data/chain_train.jsonl "
       f"{WORK}/aci_train/data/so_train.jsonl --out {ADAPTER} --save-steps 25")
    start_generator()
else:
    print("skipped: verifier already trained")
# training facts for the results file
info = {"adapter": ADAPTER, "base_model": "Qwen/Qwen2.5-3B-Instruct", "generator": PROFILE["model"], "gpu": gpu_name}
for name in ("chain_train", "so_train"):
    p = f"{WORK}/aci_train/data/{name}.jsonl"
    info[f"{name}_examples"] = sum(1 for _ in open(p)) if os.path.exists(p) else None
if os.path.exists(f"{ADAPTER}/heldout_accuracy.json"):
    info["heldout"] = json.load(open(f"{ADAPTER}/heldout_accuracy.json"))
json.dump(info, open(f"{WORK}/train_info.json", "w"), indent=1)
print(json.dumps(info, indent=1))

generator stopped
train 2875  held-out 304  labels ['chain_broken', 'chain_ok', 'contradicted', 'supported', 'unsupported']

Fetching 2 files: 100%|██████████| 2/2 [00:18<00:00,  9.25s/it]

Loading weights: 100%|██████████| 434/434 [00:01<00:00, 227.57it/s]
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607

  0%|          | 0/360 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)

  3%|▎         | 10/360 [01:45<1:01:44, 10.58s/it]
                                                  

  6%|▌         | 20/360 [03

# Part B · RQ2 on the 201 verify-and-ask cases

From here on, everything goes into the `va/` folders.

### B1 · Build the input (blocks 0 and 4, seconds)
For each case: the simulated conversation, numbered T1…Tn (block 0), and the A/P that RQ1's note writer produced, split into one statement per sentence and tagged Assessment or Plan (block 4). Filler such as "No explicit plan was documented" is dropped.

No masking is needed: the simulated conversation only takes the history, so it holds no spoken diagnosis or plan.

Expected: `va_dev` 26 cases / 119 statements, `va_test` 110 / 470. The 65 `va_trainorig` cases are written but not run.

In [21]:
use_folders("va")
if not os.path.exists(f"{WORK}/va/data/statements_va_test.jsonl"):
    sh(f"python {SCRIPTS}/build_dataset_va.py")
else:
    print("skipped: input already built")

folders -> /content/drive/MyDrive/tokenizers/rq2_evidence_chain/work/va/
[va_dev] encounters=26 statements=119 (placeholders excluded: 3)
   section: {'Assessment': 53, 'Plan': 66}
   encounters with no Plan statement: 3
[va_test] encounters=110 statements=470 (placeholders excluded: 35)
   section: {'Assessment': 233, 'Plan': 237}
   encounters with no Plan statement: 20
[va_trainorig] encounters=65 statements=282 (placeholders excluded: 23)
   section: {'Assessment': 141, 'Plan': 141}
   encounters with no Plan statement: 17
wrote /content/drive/MyDrive/tokenizers/rq2_evidence_chain/work/va/data/ (encounters, statements, audit sheets, va_manifest.json)


### B2 · Plant wrong statements (the test copy, about 10 minutes)
Three wrong A/P statements per case, the same three kinds as in training. They're mixed into a **test copy** only, so the results can measure how many are caught. They never reach the final notes. Accepted without review (`--accept-all`).

In [22]:
for split in ("va_dev", "va_test"):
    if not os.path.exists(f"{WORK}/va/data/planted_{split}.jsonl"):
        sh(f"python plant_errors.py --split {split}")
        sh(f"python plant_errors.py --split {split} --merge --accept-all")
    else:
        print(split, "skipped")

  D2N071: 3 planted
  D2N073: 3 planted
  D2N079: 3 planted
  D2N088: 3 planted
  D2N104: 3 planted
  D2N111: 3 planted
  D2N117: 3 planted
  D2N118: 3 planted
  D2N128: 3 planted
  D2N129: 3 planted
  D2N133: 3 planted
  D2N135: 3 planted
  D2N137: 3 planted
  D2N143: 3 planted
  D2N160: 3 planted
  D2N165: 3 planted
  D2N166: 3 planted
  D2N169: 3 planted
  D2N174: 3 planted
  D2N177: 3 planted
  D2N179: 3 planted
  D2N193: 3 planted
  D2N194: 3 planted
  D2N201: 3 planted
  D2N202: 3 planted
  D2N203: 3 planted
wrote 78 candidates -> /content/drive/MyDrive/tokenizers/rq2_evidence_chain/work/va/data/planted_candidates_va_dev.csv. Review 'approved' before --merge.
merged 78 planted statements: {'unsupported_diagnosis': 26, 'plan_without_evidence': 26, 'contradicted_inference': 26}
  D2N068: 3 planted
  D2N069: 3 planted
  D2N070: 3 planted
  D2N072: 3 planted
  D2N074: 3 planted
  D2N075: 3 planted
  D2N076: 3 planted
  D2N077: 3 planted
  D2N078: 3 planted
  D2N080: 3 planted
  D2N08

### B3 · Dev runs (about 1–1.5 hours)
Every condition on the 26 dev cases. **U4 runs blocks 5–11 in full.** U3 reuses U4's facts and links, so it only reruns the chain judge without passages.
These dev numbers are **not results**. They're only used to choose each condition's flag threshold in B4.

In [23]:
def run_condition(cond, split):
    k = f" --k {K_REF}" if cond == "U4" else ""
    sh(f"python run_rq2.py --condition {cond} --split {split}{k}")

for cond in CONDITIONS:
    run_condition(cond, "va_dev")

run U4_preap_ref3_link-llm_hf: 197 statements, 0 cached

Loading weights: 100%|██████████| 434/434 [00:01<00:00, 230.62it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4446.55it/s]
  25/197
  50/197
  75/197
  100/197
  125/197
  150/197
  175/197
done: /content/drive/MyDrive/tokenizers/rq2_evidence_chain/work/va/runs/va_dev/U4_preap_ref3_link-llm_hf.jsonl
run U3_preap_link-llm_hf: 197 statements, 0 cached

Loading weights: 100%|██████████| 434/434 [00:01<00:00, 228.71it/s]
  25/197
  50/197
  75/197
  100/197
  125/197
  150/197
  175/197
done: /content/drive/MyDrive/tokenizers/rq2_evidence_chain/work/va/runs/va_dev/U3_preap_link-llm_hf.jsonl
run U2_preap_hf: 197 statements, 0 cached

Loading weights: 100%|██████████| 434/434 [00:01<00:00, 227.50it/s]
  25/197
  50/197
  75/197
  100/197
  125/197
  150/197
  175/197
done: /content/drive/MyDrive/tokenizers/rq2_evidence_chain/work/va/runs/va_dev/U2_preap_hf.jsonl
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.p

### B4 · Choose the flag thresholds on dev (block 11, seconds)
For each condition, tries every cut-off and keeps the one that best balances *keeping correct statements* against *catching planted ones* (Youden's J). Saved to `reports/thresholds.json` and **frozen** for the test runs.

In [24]:
sh("python evaluate_rq2.py --split va_dev --tune")
print(json.dumps(json.load(open(f"{WORK}/va/reports/thresholds.json")), indent=1))

thresholds (dev, Youden J): {'U2_preap_hf': 0.977, 'U3_preap_link-llm_hf': 0.308, 'U4_preap_ref3_link-llm_hf': 0.283, 'U1_preap_hf+tuned': 0.414}
| run | section | retention | rej. precision | rej. recall | flag rate | AUC | s/claim |
|---|---|---|---|---|---|---|---|
| U1_preap_hf | all | 0.605 | 0.608 | 0.936 | 0.609 | 0.740 | 0.193 |
| U1_preap_hf | Assessment | 0.906 | 0.884 | 0.884 | 0.448 | 0.986 | 0.185 |
| U1_preap_hf | Plan | 0.364 | 0.455 | 1.000 | 0.762 | 0.581 | 0.200 |
| U2_preap_hf | all | 0.983 | 0.974 | 0.974 | 0.396 | 0.993 | 0.495 |
| U2_preap_hf | Assessment | 0.981 | 0.976 | 0.953 | 0.438 | 0.996 | 0.499 |
| U2_preap_hf | Plan | 0.985 | 0.972 | 1.000 | 0.356 | 0.997 | 0.491 |
| U3_preap_link-llm_hf | all | 0.916 | 0.884 | 0.974 | 0.437 | 0.975 | 0.371 |
| U3_preap_link-llm_hf | Assessment | 0.962 | 0.955 | 0.977 | 0.458 | 0.984 | 0.376 |
| U3_preap_link-llm_hf | Plan | 0.879 | 0.810 | 0.971 | 0.416 | 0.974 | 0.367 |
| U4_preap_ref3_link-llm_hf | all | 0.908 | 0.869 

### B5 · Test runs (about 4–5 hours)
The same conditions on the 110 test cases, with thresholds frozen. These give the reported results. U4 is the long one: it extracts facts and links for every case.

In [25]:
for cond in CONDITIONS:
    run_condition(cond, "va_test")

run U4_preap_ref3_link-llm_hf: 800 statements, 0 cached

Loading weights: 100%|██████████| 434/434 [00:01<00:00, 223.29it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6179.74it/s]
  25/800
  50/800
  75/800
  100/800
  125/800
  150/800
  175/800
  200/800
  225/800
  250/800
  275/800
  300/800
  325/800
  350/800
  375/800
  400/800
  425/800
  450/800
  475/800
  500/800
  525/800
  550/800
  575/800
  600/800
  625/800
  650/800
  675/800
  700/800
  725/800
  750/800
  775/800
  800/800
done: /content/drive/MyDrive/tokenizers/rq2_evidence_chain/work/va/runs/va_test/U4_preap_ref3_link-llm_hf.jsonl
run U3_preap_link-llm_hf: 800 statements, 0 cached

Loading weights: 100%|██████████| 434/434 [00:01<00:00, 226.31it/s]
  25/800
  50/800
  75/800
  100/800
  125/800
  150/800
  175/800
  200/800
  225/800
  250/800
  275/800
  300/800
  325/800
  350/800
  375/800
  400/800
  425/800
  450/800
  475/800
  500/800
  525/800
  550/800
  575/800
  600/800
  625/800
  650/800

### B6 · RQ2 metrics on test (seconds)
Retention (correct statements kept), rejection precision and recall (planted ones caught), F1, flag rate, AUC and speed, for all statements and for Assessment and Plan separately. Also the paired comparisons (U2 vs U1, U4 vs U2, U4 vs U3) and the H2 verdict.

In [26]:
sh("python evaluate_rq2.py --split va_test --setting preap")
print(open(f"{WORK}/va/reports/va_test/metrics.md").read())

| run | section | retention | rej. precision | rej. recall | flag rate | AUC | s/claim |
|---|---|---|---|---|---|---|---|
| U1_preap_hf | all | 0.579 | 0.618 | 0.970 | 0.647 | 0.725 | 0.191 |
| U1_preap_hf | Assessment | 0.944 | 0.930 | 0.956 | 0.449 | 0.975 | 0.192 |
| U1_preap_hf | Plan | 0.219 | 0.443 | 0.987 | 0.860 | 0.534 | 0.189 |
| U2_preap_hf | all | 0.947 | 0.926 | 0.948 | 0.422 | 0.986 | 0.507 |
| U2_preap_hf | Assessment | 0.987 | 0.983 | 0.934 | 0.415 | 0.996 | 0.508 |
| U2_preap_hf | Plan | 0.907 | 0.867 | 0.966 | 0.430 | 0.982 | 0.506 |
| U3_preap_link-llm_hf | all | 0.902 | 0.872 | 0.952 | 0.450 | 0.978 | 0.363 |
| U3_preap_link-llm_hf | Assessment | 0.953 | 0.939 | 0.928 | 0.432 | 0.985 | 0.379 |
| U3_preap_link-llm_hf | Plan | 0.852 | 0.807 | 0.980 | 0.469 | 0.984 | 0.345 |
| U4_preap_ref3_link-llm_hf | all | 0.906 | 0.876 | 0.945 | 0.445 | 0.978 | 3.419 |
| U4_preap_ref3_link-llm_hf | Assessment | 0.953 | 0.938 | 0.917 | 0.428 | 0.985 | 5.613 |
| U4_preap_ref3_link-

# Part C · Sign-off and patient summary
### C1 · Simulated doctor sign-off (block 12, seconds)
Uses U4 (our method) with its dev-tuned threshold. Every A/P line is marked OK or flagged. OK lines go into the **signed note**; flagged lines are held back for the doctor, never deleted or rewritten. RQ1's open questions are passed on as pending. Also counts how many planted errors would still reach the signed note.

In [27]:
U4_RUN = f"U4_preap_ref{K_REF}_link-llm_hf"
sh(f"python {SCRIPTS}/signoff.py --run {U4_RUN} --out {WORK}/final")

run U4_preap_ref3_link-llm_hf  threshold 0.2830
[va_dev] 26 cases signed
[va_test] 110 cases signed
[va_trainorig] no run file runs/va_trainorig/U4_preap_ref3_link-llm_hf.jsonl - skipped
{
 "run": "U4_preap_ref3_link-llm_hf",
 "threshold": 0.2829525,
 "rule": "flagged = P(chain broken) >= threshold; flagged lines held back for doctor review, OK lines signed",
 "splits": {
  "va_dev": {
   "encounters": 26,
   "valid_statements": 119,
   "planted_statements": 78,
   "valid_held_back": 11,
   "planted_flagged": 73,
   "planted_reaching_signed_note": 5,
   "planted_reaching_rate": 0.0641,
   "flags_per_note_test_copy": 3.231,
   "cases": 26,
   "real_ap_lines": 119,
   "real_flagged": 11,
   "real_not_checked": 0,
   "flags_per_note_real": 0.423
  },
  "va_test": {
   "encounters": 110,
   "valid_statements": 470,
   "planted_statements": 330,
   "valid_held_back": 44,
   "planted_flagged": 312,
   "planted_reaching_signed_note": 18,
   "planted_reaching_rate": 0.0545,
   "flags_per_note_

### C2 · Patient summaries (blocks 13–14, about 1 hour)
Qwen3-14B rewrites each signed note in plain language (block 13). The verifier then checks every sentence against the signed note and drops any it doesn't back up (block 14).
For **H3**, the same writer also summarises the *unverified* Stage 1 note with no checker, and both summaries are measured the same way: unsupported sentences, next-step recall and reading grade. Resumes per case.

In [28]:
sh(f"python {SCRIPTS}/patient_summary.py --final {WORK}/final")


Loading weights: 100%|██████████| 434/434 [00:01<00:00, 226.03it/s]
  [1/136] D2N068: kept 5/6 sentences
  [2/136] D2N069: kept 5/5 sentences
  [3/136] D2N070: kept 11/11 sentences
  [4/136] D2N071: kept 10/10 sentences
  [5/136] D2N072: kept 5/5 sentences
  [6/136] D2N073: kept 7/7 sentences
  [7/136] D2N074: kept 9/9 sentences
  [8/136] D2N075: kept 10/10 sentences
  [9/136] D2N076: kept 6/6 sentences
  [10/136] D2N077: kept 5/6 sentences
  [11/136] D2N078: kept 7/8 sentences
  [12/136] D2N079: kept 10/10 sentences
  [13/136] D2N080: kept 4/4 sentences
  [14/136] D2N081: kept 4/4 sentences
  [15/136] D2N082: kept 4/5 sentences
  [16/136] D2N083: kept 6/6 sentences
  [17/136] D2N084: kept 3/4 sentences
  [18/136] D2N085: kept 7/7 sentences
  [19/136] D2N087: kept 8/8 sentences
  [20/136] D2N088: kept 6/7 sentences
  [21/136] D2N089: kept 3/4 sentences
  [22/136] D2N090: kept 4/4 sentences
  [23/136] D2N091: kept 6/6 sentences
  [24/136] D2N092: kept 9/9 sentences
  [25/136] D2N093: k

# Part D · Results
### D1 · Export the results
Writes the final files to `rq2_evidence_chain/results/` on Drive, plus `rq2_evidence_chain/rq2_results.zip` to download:
- `blocks/b04_statements.csv` … `blocks/b14_summary_checks.csv`: one CSV per block, with that block's own results
- `metrics.json`: every number, grouped by block (RQ2 table, recall per error type, paired tests and H2, sign-off, H3, verifier training)
- `combined.json`: one entry per case, with everything joined

In [29]:
sh(f"python {SCRIPTS}/export_results.py --final {WORK}/final --out {RESULTS} --train-info {WORK}/train_info.json")
sh(f"cd {BASE} && rm -f rq2_results.zip && zip -qr rq2_results.zip results")
m = json.load(open(f"{RESULTS}/metrics.json"))
print("files:", json.dumps(m["files"], indent=1))
print("\nRQ2 on test (all statements):")
for r in m["b11_flagger_rq2"].get("va_test", {}).get("table", []):
    if r["section"] == "all":
        print(f"  {r['run']:<32} retention {r['valid_retention']:.3f}  recall {r['reject_recall']:.3f}  "
              f"precision {r['reject_precision']:.3f}  AUC {r['auc']}")
print("\nH2:", json.dumps((m["b11_flagger_rq2"].get("va_test", {}).get("paired_comparisons_and_H2") or {}).get("H2"), indent=1))
print(f"\nDownload: {BASE}/rq2_results.zip")

{
 "b04_statements": 1035,
 "b05_components": 1133,
 "b06_facts": 2612,
 "b07_fact_checks": 2612,
 "b08_links": 1133,
 "b09_references": 1133,
 "b10_chain_scores": 1133,
 "b11_flags": 997,
 "b12_signoff": 136,
 "b13_summary_drafts": 1997,
 "b14_summary_checks": 1997
}
wrote /content/drive/MyDrive/tokenizers/rq2_evidence_chain/results/ (blocks/*.csv, metrics.json, combined.json: 136 cases)
files: {
 "b04_statements": 1035,
 "b05_components": 1133,
 "b06_facts": 2612,
 "b07_fact_checks": 2612,
 "b08_links": 1133,
 "b09_references": 1133,
 "b10_chain_scores": 1133,
 "b11_flags": 997,
 "b12_signoff": 136,
 "b13_summary_drafts": 1997,
 "b14_summary_checks": 1997
}

RQ2 on test (all statements):
  U1_preap_hf                      retention 0.579  recall 0.970  precision 0.618  AUC 0.7246357188910381
  U2_preap_hf                      retention 0.947  recall 0.948  precision 0.926  AUC 0.985538362346873
  U3_preap_link-llm_hf             retention 0.902  recall 0.952  precision 0.872  AUC 0.9

### D2 · Finish: stop the generator and release the GPU
Stops Qwen3-14B and disconnects the runtime, so it stops using compute units. Everything is already on Drive. Set `DISCONNECT_AT_END = False` in step 0 to keep the session.

In [30]:
stop_generator()
if DISCONNECT_AT_END:
    from google.colab import runtime
    print("All done. Releasing the GPU.")
    runtime.unassign()

generator stopped
All done. Releasing the GPU.
